# ABT — Pipeline Python

Versão em PySpark das transformações do notebook **ABT (SQL)**, que constrói a ABT (Analytical Base Table) de churn a partir de `workspace.default.train`.

**Como funciona:**
* A tabela `workspace.default.train` é carregada e dividida em `df_train` (70%) e `df_test` (30%), com seed fixa.
* A classe `ABTPipeline` segue o padrão **fit / transform**:
  * `fit(df_train)` — aprende estatísticas **apenas com os dados de treino**: a mediana de `tenure` por tipo de `Contract` (equivalente ao `PERCENTILE_APPROX(...) OVER (PARTITION BY Contract)` do SQL).
  * `transform(df)` — aplica **todas** as transformações do SQL (lowercase das categóricas, `SeniorCitizen` para yes/no, imputações de `tenure` e `TotalCharges`, `comprometimentoRenda`, `avgMonthlyCharges`, `evolucaoMonthlyCharges`, `ai_classify` do feedback e `Churn` binário) e devolve a ABT.
* O mesmo pipeline ajustado em `df_train` é replicável em `df_test` e em **novos dados** com o mesmo schema (a coluna `Churn` é opcional — útil para dados de scoring).

In [0]:
# Carga da tabela e divisão em treino/teste
df = spark.read.table("workspace.default.train")

df_train, df_test = df.randomSplit(weights=[0.7, 0.3], seed=42)

print(f"df_train: {df_train.count()} linhas")
print(f"df_test:  {df_test.count()} linhas")

display(df_train.limit(5))

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F


class ABTPipeline:
    """Réplica em PySpark das transformações do ABT (SQL).

    * ``fit(df)`` — aprende estatísticas APENAS com os dados de treino:
      a mediana de ``tenure`` por tipo de ``Contract``.
    * ``transform(df)`` — aplica todas as transformações (limpeza, imputações,
      features derivadas e ``ai_classify`` do feedback) e devolve a ABT.
      Replicável em ``df_test`` e em novos dados com o schema da tabela
      ``workspace.default.train`` (a coluna ``Churn`` é opcional).

    Observações:
    * Contrato não visto no ``fit`` recebe ``mediana_tenure_contrato`` nula.
    * ``ai_classify`` é amostral (pode variar entre execuções) e tem custo
      de LLM por linha — persista o resultado para não reexecutá-la.
    """

    _CATEGORICAS = [
        "Partner", "Dependents", "PhoneService", "MultipleLines",
        "InternetService", "OnlineSecurity", "OnlineBackup",
        "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies",
        "Contract", "PaperlessBilling", "PaymentMethod",
    ]

    # Colunas do SELECT final do ABT (SQL), na mesma ordem
    _COLUNAS_ABERTA = [
        "customerID", "gender", "SeniorCitizen", "Partner", "Dependents",
        "PhoneService", "MultipleLines", "InternetService", "OnlineSecurity",
        "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV",
        "StreamingMovies", "Contract", "PaperlessBilling", "PaymentMethod",
        "tenure", "mediana_tenure_contrato", "MonthlyCharges", "TotalCharges",
        "MonthlyIncome", "comprometimentoRenda", "avgMonthlyCharges",
        "evolucaoMonthlyCharges", "motivo_feedback", "Churn",
    ]

    _EXPR_AI_CLASSIFY = """
    ai_classify(
        CustomerFeedback,
        '{
            "preco_alto": "Cliente acha o custo mensal alto ou caro em relação ao mercado, menciona preço mensal, cobranças, orçamento",
            "qualidade_servico": "Quedas frequentes, lentidão, serviço de internet não confiável ou que não atende expectativas",
            "flexibilidade_contrato": "Insatisfação com o tipo/vínculo do contrato, queria contrato mais flexível ou sem fidelidade",
            "metodo_pagamento": "Insatisfação com o método de pagamento disponível (cheque eletrônico, cartão, débito automático)",
            "financeiro_pessoal": "Problemas financeiros pessoais, mudança de endereço ou motivos pessoais alheios à empresa",
            "suporte_atendimento": "Insatisfação com atendimento, suporte técnico ou customer service",
            "satisfeito": "O cliente está satisfeito com o serviço, valor do serviço, qualidade do serviço, não tem reclamações."
        }',
        MAP(
            'version', '2.1',
            'instructions', 'Textos são feedbacks em inglês de clientes. Alguns deles cancelaram o serviço de internet e/ou telefone. Se o feedback for de satisfação com o serviço, classifique-o dessa forma. Caso contrário, classifique o MOTIVO PRIMÁRIO da insatisfação. Se, no texto, o clinte indicar que optou por cancelar ou mudar para outra empresa, classifique o MOTIVO PRIMÁRIO dessa decisão.'
        )
    ):response[0].value::STRING
    """

    def __init__(self):
        self.medianas_tenure_contrato = None

    # ---------------- API pública ----------------

    def fit(self, df: DataFrame) -> "ABTPipeline":
        """Aprende as medianas de tenure por Contract usando apenas ``df``."""
        base = self._tb_fs_sem_classificacao(df)
        linhas = (
            base.filter(F.col("tenure").isNotNull())
            .groupBy("Contract")
            .agg(F.percentile_approx("tenure", 0.5).alias("mediana"))
            .collect()
        )
        self.medianas_tenure_contrato = {
            r["Contract"]: r["mediana"] for r in linhas if r["Contract"] is not None
        }
        return self

    def transform(self, df: DataFrame) -> DataFrame:
        """Aplica as transformações do ABT e devolve o DataFrame final."""
        if self.medianas_tenure_contrato is None:
            raise RuntimeError("Chame .fit(df_train) antes de .transform(df).")

        # mediana por contrato, aprendida no fit (evita vazamento do teste)
        mapa_medianas = F.create_map(
            [F.lit(v) for kv in self.medianas_tenure_contrato.items() for v in kv]
        )

        return (
            self._tb_fs(df)
            .withColumn("mediana_tenure_contrato", mapa_medianas[F.col("Contract")])
            .withColumn(
                "avgMonthlyCharges",
                F.when(F.col("tenure") == 0, F.col("MonthlyCharges")).otherwise(
                    F.col("TotalCharges") / F.col("tenure")
                ),
            )
            .withColumn(
                "evolucaoMonthlyCharges",
                F.col("MonthlyCharges") / F.col("avgMonthlyCharges"),
            )
            .filter(F.col("tenure").isNotNull())  # WHERE tenure IS NOT NULL
            .select(*self._COLUNAS_ABERTA)
        )

    def fit_transform(self, df: DataFrame) -> DataFrame:
        return self.fit(df).transform(df)

    # ---------------- transformações internas ----------------

    def _tb_fs(self, df: DataFrame) -> DataFrame:
        """tb_fs completa do SQL (inclui a classificação do feedback)."""
        return self._tb_fs_sem_classificacao(df).withColumn(
            "motivo_feedback", F.expr(self._EXPR_AI_CLASSIFY)
        )

    def _tb_fs_sem_classificacao(self, df: DataFrame) -> DataFrame:
        """tb_fs do SQL sem o ai_classify (barata — usada também pelo fit)."""
        churn = (
            F.when(F.lower("Churn") == "yes", F.lit(1)).otherwise(F.lit(0))
            if "Churn" in df.columns
            else F.lit(None).cast("int")
        )
        return df.select(
            "customerID",
            F.lower("gender").alias("gender"),
            F.when(F.col("SeniorCitizen") == 1, "yes").otherwise("no").alias("SeniorCitizen"),
            *[F.lower(c).alias(c) for c in self._CATEGORICAS],
            # tenure: imputa 'unknown' com round(TotalCharges / MonthlyCharges)
            F.when(
                F.lower("tenure") == "unknown",
                F.when(
                    F.col("MonthlyCharges").isNotNull() & F.col("TotalCharges").isNotNull(),
                    F.round(F.col("TotalCharges") / F.col("MonthlyCharges")),
                ),
            )
            .otherwise(F.lower("tenure").cast("int"))
            .alias("tenure"),
            "MonthlyCharges",
            # TotalCharges: imputa nulo com tenure * MonthlyCharges
            F.when(
                F.col("TotalCharges").isNull(),
                F.when(
                    F.col("MonthlyCharges").isNotNull() & (F.lower("tenure") != "unknown"),
                    F.col("tenure").cast("int") * F.col("MonthlyCharges"),
                ),
            )
            .otherwise(F.col("TotalCharges"))
            .alias("TotalCharges"),
            "MonthlyIncome",
            (F.col("MonthlyCharges") / F.col("MonthlyIncome")).alias("comprometimentoRenda"),
            churn.alias("Churn"),
        )

In [0]:
# Ajusta o pipeline com df_train e replica as transformações em df_train e df_test
pipeline = ABTPipeline().fit(df_train)

print("Medianas de tenure por contrato (aprendidas apenas com df_train):")
for contrato, mediana in sorted(pipeline.medianas_tenure_contrato.items()):
    print(f"  {contrato}: {mediana}")

abt_train = pipeline.transform(df_train)
abt_test = pipeline.transform(df_test)

print(f"abt_train: {abt_train.count()} linhas | abt_test: {abt_test.count()} linhas")

display(abt_train.limit(5))

In [0]:
# Exemplo: o MESMO pipeline (já ajustado em df_train) aplicado a novos dados.
# Troque a fonte abaixo pelos seus dados reais de produção/scoring
# (ex.: spark.read.table("workspace.default.test"), quando existir).
novos_dados = spark.read.table("workspace.default.train").limit(50)

abt_novos = pipeline.transform(novos_dados)

display(abt_novos.limit(5))

## Notas

* **Paridade com o SQL:** `ABTPipeline().fit_transform(spark.read.table("workspace.default.train"))` reproduz a `workspace.default.abt` (mesmas imputações, features e medianas). Com `df_train`, a mediana é aprendida só no treino — o que evita vazamento de dados de teste no modelo.
* **`ai_classify` tem custo por linha e é amostral** (o mesmo texto pode receber rótulos distintos entre execuções). Execute uma vez e persista o resultado, por exemplo:
  * `abt_train.write.mode("overwrite").saveAsTable("workspace.default.abt_train")`
* **Contratos não vistos no `fit`** recebem `mediana_tenure_contrato` nula.
* `motivo_feedback` usa as mesmas 7 categorias do SQL: `preco_alto`, `qualidade_servico`, `flexibilidade_contrato`, `metodo_pagamento`, `financeiro_pessoal`, `suporte_atendimento`, `satisfeito`.